# Replacement-Dynamics Model with Full Market Calibration from New Vehicle Registrations

This notebook keeps the **replacement-dynamics** logic, but improves the calibration of the **future entry-share curve** by using the full external registrations dataset instead of calibrating only the EV line.

## Why we are changing the model

The previous notebook used the external registrations file mainly to calibrate the future `electric` entry share. That was already better than relying only on the internal SAAQ extrapolation, but it still left the rest of the vehicle mix mostly driven by the internal model.

That can create an inconsistency:

- `electric` is externally constrained,
- but `hev_sedan`, `hev_suv`, `ice_sedan`, `ice_suv`, and `ice_van/pickup` still move mostly according to the internal adoption rules.

If we want a more realistic adoption path, it is better to calibrate the **whole future sales mix**.

## New idea in this notebook

We use the full quarterly registrations dataset from:

- [Fig1-NMVRegist.xlsx](/Users/natomanzolli/Downloads/Fig1-NMVRegist.xlsx)

and map it into the six categories used by the replacement model:

- `electric`
- `hev_sedan`
- `hev_suv`
- `ice_sedan`
- `ice_suv`
- `ice_van/pickup`

Then we build a future benchmark for the **entire entry-share vector**, not only for EVs.

## Mapping logic

The external workbook is richer than the internal taxonomy because it distinguishes:

- total vehicle type
- passenger cars
- multi-purpose vehicles
- pickup trucks
- vans
- fuel types such as battery electric, plug-in hybrid, hybrid, diesel, gasoline, and other fuels

We map those rows into the model categories as follows:

- `electric`: total **Battery electric** share
- `hev_sedan`: **Passenger cars** with `Plug-in hybrid electric + Hybrid electric`
- `hev_suv`: **Multi-purpose vehicles** with `Plug-in hybrid electric + Hybrid electric`
- `ice_sedan`: **Passenger cars** with `Gasoline + Diesel + Other fuel types`
- `ice_suv`: **Multi-purpose vehicles** with `Gasoline + Diesel + Other fuel types`
- `ice_van/pickup`: the residual share needed to make the six categories sum to 1, which effectively captures the pickup and van classes

That last point matters: the internal taxonomy does not have separate electrified pickup or van categories, so the pickup/van market is absorbed into `ice_van/pickup`. The notebook calls this out explicitly because it is a modeling limitation, not a hidden assumption.

## Forecasting logic

1. Build the historical SAAQ replacement model with explicit entries and disposals.
2. Build external annual benchmark shares for **all six categories** from 2017 to 2025 Q1.
3. Use the observed external benchmark directly for 2021 to 2025.
4. From 2026 onward, extrapolate the benchmark using **annual share changes** rather than raw percentage growth.
5. Dampen those annual share changes through time so the mix evolves realistically instead of exploding.
6. Replace the model's future entry-share vector with this calibrated market benchmark.
7. Keep the exit-share and stock-replacement mechanics from the SAAQ model.

## Why this is more realistic

This changes the interpretation from:

- “only EV adoption is externally calibrated”

to:

- “the whole new-sales market mix is externally calibrated, while the fleet still changes gradually because of replacement dynamics.”

That is closer to how the market actually works: the sales mix can move quickly, but the fleet stock reacts more slowly.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

PROJECT_DIR = Path('/Users/natomanzolli/Documents/GitHub/MATSim-agent-vehicle-assignment/adoption prediction model')
CACHE_DIR = PROJECT_DIR / '.cache'
EXTERNAL_FILE = Path('/Users/natomanzolli/Downloads/Fig1-NMVRegist.xlsx')
POPULATION_REFERENCE_FILE = Path('/Users/natomanzolli/Downloads/1710000901-eng.csv')


## Settings

These parameters control the calibration.

### Main choices

- `CALIBRATION_WINDOW`
  Uses either the full benchmark history or only the more recent years to estimate future share changes.

- `RECENT_START_YEAR`
  Defines what “recent” means when `CALIBRATION_WINDOW = 'recent'`.

- `DAMPING_END_FACTOR`
  Shrinks future share changes over time. A smaller number means more conservative long-run change.

- `MAX_ELECTRIC_ENTRY_SHARE`
  Keeps the EV sales share from becoming unrealistically high by 2035.

- `TURNOVER_MODEL`
  Controls how many vehicles are replaced each year. This is estimated from the SAAQ entry side.

- `POPULATION_GROWTH_WINDOW`
  Controls which years from the population reference file are used to estimate the long-run net fleet growth rate.

This is the main modeling change: we no longer let the independent exit-rate fit determine total fleet collapse. Instead, net fleet change follows the population reference, while turnover still comes from the SAAQ data.


In [ ]:
ENTRY_FLAG = 'Entrant'
EXIT_FLAG = 'Sortant'
FORECAST_END_YEAR = 2035
TURNOVER_MODEL = 'average'  # 'average' or 'linear'
TURNOVER_RATE_CLIP = (0.03, 0.12)
SHARE_DELTA_CLIP = (-0.05, 0.05)

CALIBRATION_WINDOW = 'recent'  # 'recent' or 'full'
RECENT_START_YEAR = 2021
POPULATION_GROWTH_WINDOW = (2021, 2026)
DAMPING_END_FACTOR = 0.35
MAX_ELECTRIC_ENTRY_SHARE = 0.45

vehicle_cols = ['electric', 'hev_sedan', 'hev_suv', 'ice_sedan', 'ice_suv', 'ice_van/pickup']
quarter_order = ['T1', 'T2', 'T3', 'T4']


## Step 1: Read and Reshape the External Quarterly Registrations Dataset

The workbook uses a two-row year/quarter header and a stacked block structure by vehicle body class and fuel type.

We reshape it into a long table with:

- `body_class`
- `fuel_type`
- `year`
- `quarter`
- `registrations`

This makes the later category mapping explicit and easy to audit.

In [ ]:
raw_external = pd.read_excel(EXTERNAL_FILE, sheet_name='Sheet 1', header=None)

years = raw_external.iloc[0, 2:].ffill().astype(int).tolist()
quarters = raw_external.iloc[1, 2:].tolist()

records = []
current_body_class = None
for row_idx in range(2, len(raw_external)):
    row = raw_external.iloc[row_idx]
    if pd.notna(row[0]):
        current_body_class = str(row[0]).strip()

    fuel_type = row[1]
    if pd.isna(fuel_type):
        continue

    fuel_type = str(fuel_type).strip()
    values = row.iloc[2:].tolist()

    for year, quarter, value in zip(years, quarters, values):
        if pd.isna(value):
            continue
        records.append({
            'body_class': current_body_class,
            'fuel_type': fuel_type,
            'year': int(year),
            'quarter': str(quarter),
            'registrations': float(value),
        })

external_long = pd.DataFrame(records)
external_long.head()

## Step 2: Map the External Dataset into the Model's Six Vehicle Categories

This is the core improvement.

Instead of extracting only an EV curve, we build a full category vector from the registrations file.

### Category mapping used here

- `electric`
  All **Battery electric** registrations, across body classes.

- `hev_sedan`
  **Passenger cars** with `Plug-in hybrid electric + Hybrid electric`.

- `hev_suv`
  **Multi-purpose vehicles** with `Plug-in hybrid electric + Hybrid electric`.

- `ice_sedan`
  **Passenger cars** with `Gasoline + Diesel + Other fuel types`.

- `ice_suv`
  **Multi-purpose vehicles** with `Gasoline + Diesel + Other fuel types`.

- `ice_van/pickup`
  Residual share after the five categories above are removed from the total.

This residual treatment is deliberate. It lets the full external table inform the calibration while respecting the internal six-category structure.

In [ ]:
quarterly_cube = (
    external_long.pivot_table(
        index=['year', 'quarter'],
        columns=['body_class', 'fuel_type'],
        values='registrations',
        aggfunc='sum',
        fill_value=0,
    )
    .sort_index()
)

quarterly_cube.columns = pd.MultiIndex.from_tuples(quarterly_cube.columns)


def qcol(body_class, fuel_type):
    if (body_class, fuel_type) in quarterly_cube.columns:
        return quarterly_cube[(body_class, fuel_type)]
    return pd.Series(0.0, index=quarterly_cube.index)

external_mapped_counts = pd.DataFrame(index=quarterly_cube.index)
external_mapped_counts['electric'] = qcol('Total, vehicle type', 'Battery electric')
external_mapped_counts['hev_sedan'] = (
    qcol('Passenger cars', 'Plug-in hybrid electric')
    + qcol('Passenger cars', 'Hybrid electric')
)
external_mapped_counts['hev_suv'] = (
    qcol('Multi-purpose vehicles', 'Plug-in hybrid electric')
    + qcol('Multi-purpose vehicles', 'Hybrid electric')
)
external_mapped_counts['ice_sedan'] = (
    qcol('Passenger cars', 'Gasoline')
    + qcol('Passenger cars', 'Diesel')
    + qcol('Passenger cars', 'Other fuel types')
)
external_mapped_counts['ice_suv'] = (
    qcol('Multi-purpose vehicles', 'Gasoline')
    + qcol('Multi-purpose vehicles', 'Diesel')
    + qcol('Multi-purpose vehicles', 'Other fuel types')
)

external_mapped_counts['total_registrations'] = quarterly_cube.xs('Total, vehicle type', axis=1, level=0).sum(axis=1)
external_mapped_counts['ice_van/pickup'] = (
    external_mapped_counts['total_registrations']
    - external_mapped_counts[['electric', 'hev_sedan', 'hev_suv', 'ice_sedan', 'ice_suv']].sum(axis=1)
).clip(lower=0)

external_mapped_counts = external_mapped_counts.reset_index()
external_mapped_counts['quarter_num'] = external_mapped_counts['quarter'].map({q: i + 1 for i, q in enumerate(quarter_order)})
external_mapped_counts['period_label'] = external_mapped_counts['year'].astype(str) + '-' + external_mapped_counts['quarter']

external_mapped_shares = external_mapped_counts.copy()
for col in vehicle_cols:
    external_mapped_shares[col] = external_mapped_shares[col] / external_mapped_shares['total_registrations']

external_mapped_shares[['year', 'quarter', *vehicle_cols, 'total_registrations']].head(12)

## Step 3: Build a Smoother Annual Benchmark up to 2025 Q1

The external dataset is quarterly and ends in `2025 Q1`.

To avoid overreacting to one quarter, we build:

- annual observed shares for 2017 to 2024,
- a **trailing four-quarter** share for 2025,

for **all six categories**.

This gives us a coherent all-category benchmark at the point where the forecast starts to rely more heavily on extrapolation.

In [ ]:
external_trailing = external_mapped_counts.copy().sort_values(['year', 'quarter_num']).reset_index(drop=True)
for col in vehicle_cols + ['total_registrations']:
    external_trailing[f'{col}_trailing4'] = external_trailing[col].rolling(4, min_periods=4).sum()

external_trailing_shares = external_trailing[['year', 'quarter']].copy()
for col in vehicle_cols:
    external_trailing_shares[f'{col}_trailing4_share'] = (
        external_trailing[f'{col}_trailing4'] / external_trailing['total_registrations_trailing4']
    )

external_annual_counts = external_mapped_counts.groupby('year')[vehicle_cols + ['total_registrations']].sum()
external_annual_benchmark = external_annual_counts[vehicle_cols].div(external_annual_counts['total_registrations'], axis=0)

latest_trailing_row = external_trailing_shares.loc[
    (external_trailing_shares['year'] == 2025) & (external_trailing_shares['quarter'] == 'T1')
].iloc[0]
for col in vehicle_cols:
    external_annual_benchmark.loc[2025, col] = float(latest_trailing_row[f'{col}_trailing4_share'])

external_annual_benchmark.loc[2017:2025]

## Step 4: Visual Check of the External Market Benchmark

This plot lets us verify that the benchmark behaves sensibly before it affects the fleet model.

We should see:

- `electric` increasing,
- hybrid categories changing in a smoother way,
- ICE categories trending down rather than collapsing abruptly.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for col in vehicle_cols:
    ax.plot(external_annual_benchmark.loc[2017:2025].index, external_annual_benchmark.loc[2017:2025, col], marker='o', label=col)
ax.set_title('External Annual New-Vehicle Benchmark Shares Mapped to Model Categories')
ax.set_xlabel('Year')
ax.set_ylabel('Share of new registrations')
ax.legend(ncol=2)
plt.show()

## Step 5: Load the Historical SAAQ Fleet, Entry, and Exit Data

The external workbook calibrates the **sales mix**.

The SAAQ cache still provides the stock dynamics:

- fleet composition,
- entries,
- disposals,
- and therefore the gradual replacement mechanism that translates sales changes into fleet changes.

In [ ]:
fleet_counts = pd.read_pickle(CACHE_DIR / 'saq_vehicle_counts.pkl')
entry_counts = pd.read_pickle(CACHE_DIR / 'saq_entry_counts.pkl')
entry_full = pd.read_pickle(CACHE_DIR / 'saq_entry_full.pkl')

fleet_pivot = (
    fleet_counts.pivot_table(
        index='AnneeSAAQ',
        columns='vehicle_type',
        values='count',
        aggfunc='sum',
        fill_value=0,
    )
    .reindex(columns=vehicle_cols, fill_value=0)
    .sort_index()
)
fleet_pivot['total_vehicles'] = fleet_pivot.sum(axis=1)

entry_pivot = (
    entry_counts.pivot_table(
        index='AnneeSAAQ',
        columns='vehicle_type',
        values=ENTRY_FLAG,
        aggfunc='sum',
        fill_value=0,
    )
    .reindex(columns=vehicle_cols, fill_value=0)
    .sort_index()
)
entry_pivot['total_entries'] = entry_pivot.sum(axis=1)

exit_pivot = (
    entry_full.pivot_table(
        index='AnneeSAAQ',
        columns='vehicle_type',
        values=EXIT_FLAG,
        aggfunc='sum',
        fill_value=0,
    )
    .reindex(columns=vehicle_cols, fill_value=0)
    .sort_index()
)
exit_pivot['total_exits'] = exit_pivot.sum(axis=1)

fleet_pivot.head()

## Step 6: Compare the Internal SAAQ Entry Shares with the External Benchmark

This comparison is helpful because it tells us whether the external registrations series is broadly aligned with the internal SAAQ structure.

We do not expect a perfect match because the taxonomies are not identical, but the broad direction should be consistent.

In [ ]:
entry_share_hist = entry_pivot[vehicle_cols].div(entry_pivot['total_entries'].replace(0, np.nan), axis=0).fillna(0)
exit_share_hist = exit_pivot[vehicle_cols].div(exit_pivot['total_exits'].replace(0, np.nan), axis=0).fillna(0)

comparison_years = sorted(set(entry_share_hist.index).intersection(set(external_annual_benchmark.index)))
share_alignment = pd.DataFrame(index=comparison_years)
for col in vehicle_cols:
    share_alignment[f'saq_{col}'] = entry_share_hist.loc[comparison_years, col]
    share_alignment[f'external_{col}'] = external_annual_benchmark.loc[comparison_years, col]
    share_alignment[f'diff_{col}'] = share_alignment[f'external_{col}'] - share_alignment[f'saq_{col}']

share_alignment.round(4).head()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 12), sharex=True)
axes = axes.flatten()
for ax, col in zip(axes, vehicle_cols):
    ax.plot(comparison_years, entry_share_hist.loc[comparison_years, col], marker='o', label='SAAQ entry share')
    ax.plot(comparison_years, external_annual_benchmark.loc[comparison_years, col], marker='s', label='External benchmark')
    ax.set_title(col)
    ax.set_ylabel('Share')
    ax.legend()
plt.suptitle('Historical Alignment: SAAQ Entry Share vs External Registrations Benchmark', y=1.02)
plt.tight_layout()
plt.show()

## Step 7: Use Population Growth for Net Fleet Change and SAAQ for Turnover

This is the key correction to the stock-flow model.

In the earlier version, entry and exit rates were projected independently. That caused the exit path to stay well above the entry path, which mechanically forced the total fleet to collapse.

Here we separate two ideas:

- **turnover**: how many vehicles are replaced each year
- **net growth**: how much the total fleet changes from one year to the next

### New rule used here

- Turnover is estimated from the SAAQ `Entrant` history.
- Net fleet growth follows the Quebec population reference from:
  - [1710000901-eng.csv](/Users/natomanzolli/Downloads/1710000901-eng.csv)

For `2021` to `2026`, we use the observed annual Q1 population growth from the reference file.

For `2027` onward, we use the average population growth over the selected calibration window.

This keeps the total fleet path smooth and realistic, while still letting the vehicle mix evolve through replacement.


In [ ]:
flow_rates = pd.DataFrame(index=fleet_pivot.index)
flow_rates['fleet_prev_year'] = fleet_pivot['total_vehicles'].shift(1)
flow_rates['total_entries'] = entry_pivot['total_entries'].reindex(flow_rates.index)
flow_rates['total_exits'] = exit_pivot['total_exits'].reindex(flow_rates.index)
flow_rates['entry_rate'] = (flow_rates['total_entries'] / flow_rates['fleet_prev_year']).replace([np.inf, -np.inf], np.nan)
flow_rates['exit_rate'] = (flow_rates['total_exits'] / flow_rates['fleet_prev_year']).replace([np.inf, -np.inf], np.nan)

flow_source = flow_rates.dropna().copy()
last_observed_year = int(fleet_pivot.index.max())
future_years = list(range(last_observed_year + 1, FORECAST_END_YEAR + 1))

population_raw = pd.read_csv(POPULATION_REFERENCE_FILE, skiprows=7)
population_quebec = population_raw[population_raw['Geography'] == 'Quebec'].copy()
population_quebec = population_quebec.drop(columns=[c for c in population_quebec.columns if str(c).startswith('Unnamed')], errors='ignore')
pop_row = population_quebec.iloc[0].drop(labels=['Geography'])
pop_series = pop_row.astype(str).str.replace(',', '', regex=False)
pop_series = pd.to_numeric(pop_series, errors='coerce').dropna()

population_q1 = {}
for col, value in pop_series.items():
    col = str(col)
    if col.startswith('Q1 '):
        population_q1[int(col.split()[-1])] = float(value)
population_q1 = pd.Series(population_q1).sort_index()
population_growth = population_q1.pct_change()

start_year, end_year = POPULATION_GROWTH_WINDOW
reference_population_growth = float(population_growth.loc[start_year:end_year].mean())
future_population_growth = pd.Series(index=future_years, dtype=float)
for year in future_years:
    if year in population_growth.index and pd.notna(population_growth.loc[year]):
        future_population_growth.loc[year] = float(population_growth.loc[year])
    else:
        future_population_growth.loc[year] = reference_population_growth

def fit_turnover_projection(series, future_years, clip_bounds):
    hist = series.dropna().copy()
    if len(hist) < 2:
        return pd.Series({year: float(hist.iloc[-1]) if len(hist) else 0.0 for year in future_years})

    x = hist.index.to_numpy(dtype=float)
    y = hist.to_numpy(dtype=float)
    slope, intercept = np.polyfit(x, y, 1)
    pred = pd.Series({year: intercept + slope * year for year in future_years}, dtype=float)
    return pred.clip(lower=clip_bounds[0], upper=clip_bounds[1])

if TURNOVER_MODEL == 'linear':
    future_turnover_rate = fit_turnover_projection(flow_source['entry_rate'], future_years, TURNOVER_RATE_CLIP)
else:
    future_turnover_rate = pd.Series(flow_source['entry_rate'].mean(), index=future_years, dtype=float).clip(lower=TURNOVER_RATE_CLIP[0], upper=TURNOVER_RATE_CLIP[1])

target_total_fleet = pd.Series(index=[last_observed_year] + future_years, dtype=float)
target_total_fleet.loc[last_observed_year] = float(fleet_pivot.loc[last_observed_year, 'total_vehicles'])
for year in future_years:
    target_total_fleet.loc[year] = target_total_fleet.loc[year - 1] * (1 + float(future_population_growth.loc[year]))

population_reference_summary = pd.DataFrame({
    'population_q1': population_q1,
    'population_growth': population_growth,
}).loc[2015:2026]

future_stock_controls = pd.DataFrame({
    'future_turnover_rate': future_turnover_rate,
    'future_population_growth': future_population_growth,
    'target_total_fleet': target_total_fleet.loc[future_years],
})

future_stock_controls.round(4)


## Step 8: Build the Base Internal Future Shares

This is the internal no-calibration baseline.

It lets us compare:

- the original SAAQ-only future path,
- and the externally calibrated full-market path.

In [ ]:
entry_share_delta = entry_share_hist.diff().replace([np.inf, -np.inf], 0).fillna(0)
exit_share_delta = exit_share_hist.diff().replace([np.inf, -np.inf], 0).fillna(0)
entry_share_delta = entry_share_delta.clip(lower=SHARE_DELTA_CLIP[0], upper=SHARE_DELTA_CLIP[1])
exit_share_delta = exit_share_delta.clip(lower=SHARE_DELTA_CLIP[0], upper=SHARE_DELTA_CLIP[1])

avg_entry_share_delta = entry_share_delta.mean().reindex(vehicle_cols).fillna(0)
avg_exit_share_delta = exit_share_delta.mean().reindex(vehicle_cols).fillna(0)

base_entry_share_future = entry_share_hist.copy()
base_exit_share_future = exit_share_hist.copy()

current_entry_share = base_entry_share_future.loc[last_observed_year, vehicle_cols].astype(float).copy()
current_exit_share = base_exit_share_future.loc[last_observed_year, vehicle_cols].astype(float).copy()
current_entry_share = current_entry_share / current_entry_share.sum()
current_exit_share = current_exit_share / current_exit_share.sum()

for year in future_years:
    next_entry_share = (current_entry_share + avg_entry_share_delta).clip(lower=0)
    next_exit_share = (current_exit_share + avg_exit_share_delta).clip(lower=0)

    next_entry_share = next_entry_share / next_entry_share.sum() if next_entry_share.sum() > 0 else current_entry_share.copy()
    next_exit_share = next_exit_share / next_exit_share.sum() if next_exit_share.sum() > 0 else current_exit_share.copy()

    base_entry_share_future.loc[year, vehicle_cols] = next_entry_share
    base_exit_share_future.loc[year, vehicle_cols] = next_exit_share
    current_entry_share = next_entry_share.copy()
    current_exit_share = next_exit_share.copy()

base_entry_share_future.loc[2021:2035, vehicle_cols].round(4).head()

## Step 9: Build the External Full-Market Calibration Path

Here we use the full benchmark for all categories.

### How the future path is constructed

- For `2021` to `2025`, we use the observed external benchmark directly.
- For `2026` onward, we compute the average annual **share change** for each category from the benchmark history.
- We damp those annual changes through time.
- We clip negative values to zero and renormalize so the row still sums to 1.

This is the key change relative to the previous notebook: the entire future entry-share vector is calibrated, not only the EV line.

In [ ]:
if CALIBRATION_WINDOW == 'recent':
    calibration_source = external_annual_benchmark.loc[RECENT_START_YEAR:2025].copy()
else:
    calibration_source = external_annual_benchmark.loc[2017:2025].copy()

benchmark_share_delta = calibration_source[vehicle_cols].diff().replace([np.inf, -np.inf], 0).fillna(0)
benchmark_share_delta = benchmark_share_delta.clip(lower=SHARE_DELTA_CLIP[0], upper=SHARE_DELTA_CLIP[1])
avg_benchmark_share_delta = benchmark_share_delta.mean().reindex(vehicle_cols).fillna(0)

calibrated_entry_share_future = base_entry_share_future.copy()
for year in range(2021, min(2025, FORECAST_END_YEAR) + 1):
    calibrated_entry_share_future.loc[year, vehicle_cols] = external_annual_benchmark.loc[year, vehicle_cols].astype(float)

current_external_share = external_annual_benchmark.loc[2025, vehicle_cols].astype(float).copy()
for position, year in enumerate(range(2026, FORECAST_END_YEAR + 1), start=1):
    damping = 1 - (1 - DAMPING_END_FACTOR) * (position / max(1, FORECAST_END_YEAR - 2025))
    next_share = (current_external_share + avg_benchmark_share_delta * damping).clip(lower=0)
    if next_share.sum() > 0:
        next_share = next_share / next_share.sum()
    else:
        next_share = current_external_share.copy()

    next_share['electric'] = min(float(next_share['electric']), MAX_ELECTRIC_ENTRY_SHARE)
    non_electric = next_share.drop('electric')
    if non_electric.sum() > 0:
        next_share.loc[non_electric.index] = non_electric * ((1 - next_share['electric']) / non_electric.sum())

    calibrated_entry_share_future.loc[year, vehicle_cols] = next_share
    current_external_share = next_share.copy()

full_market_entry_share_comparison = pd.concat(
    {
        'base': base_entry_share_future.loc[2021:2035, vehicle_cols],
        'calibrated': calibrated_entry_share_future.loc[2021:2035, vehicle_cols],
    },
    axis=1,
)
full_market_entry_share_comparison.head()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 12), sharex=True)
axes = axes.flatten()
plot_years = list(range(2021, FORECAST_END_YEAR + 1))
for ax, col in zip(axes, vehicle_cols):
    ax.plot(plot_years, base_entry_share_future.loc[plot_years, col], label='Base internal path', marker='o')
    ax.plot(plot_years, calibrated_entry_share_future.loc[plot_years, col], label='Calibrated market path', marker='s')
    if col in external_annual_benchmark.columns:
        external_years = [year for year in plot_years if year in external_annual_benchmark.index]
        ax.plot(external_years, external_annual_benchmark.loc[external_years, col], linestyle='--', color='black', label='Observed external benchmark')
    ax.set_title(col)
    ax.set_ylabel('Entry share')
    ax.legend()
plt.suptitle('Base vs Calibrated Future Entry Shares by Vehicle Category', y=1.02)
plt.tight_layout()
plt.show()

## Step 9B: Check the Population-Linked Stock Controls

Before rebuilding the fleet, we inspect the controls driving the total stock path:

- `future_turnover_rate`: how many vehicles are replaced each year
- `future_population_growth`: the net stock growth anchor from the population reference
- `target_total_fleet`: the implied total fleet path


In [ ]:
future_stock_controls.round(4)


## Step 10: Rebuild the Fleet Under Both Scenarios

We now compare two versions of the same replacement model:

- **Base**: future entry shares come only from the internal SAAQ trend.
- **Calibrated**: future entry shares come from the full external market benchmark.

Everything else remains the same:

- the historical fleet,
- the projected entry and exit rates,
- the exit-share path,
- and the stock accounting.

In [ ]:
def rebuild_fleet(
    fleet_hist,
    entry_share_future,
    exit_share_future,
    future_turnover_rate,
    target_total_fleet,
    vehicle_cols,
    forecast_end_year,
):
    projected_counts = fleet_hist.copy()
    current_counts = projected_counts.loc[int(fleet_hist.index.max()), vehicle_cols].astype(float).copy()
    projection_rows = []

    for year in range(int(fleet_hist.index.max()) + 1, forecast_end_year + 1):
        current_total = float(current_counts.sum())
        turnover_rate = float(future_turnover_rate.loc[year])
        target_total = float(target_total_fleet.loc[year])

        entries = current_total * turnover_rate
        exits = max(current_total + entries - target_total, 0.0)

        additions = entries * entry_share_future.loc[year, vehicle_cols].astype(float)
        removals = exits * exit_share_future.loc[year, vehicle_cols].astype(float)

        next_counts = (current_counts + additions - removals).clip(lower=0)
        next_total = float(next_counts.sum())

        if next_total > 0 and target_total > 0:
            next_counts = next_counts * (target_total / next_total)
            next_total = float(next_counts.sum())

        projected_counts.loc[year, vehicle_cols] = next_counts
        projected_counts.loc[year, 'total_vehicles'] = next_total
        projection_rows.append({
            'year': year,
            'turnover_rate': turnover_rate,
            'population_growth': float(future_population_growth.loc[year]),
            'entries': entries,
            'exits': exits,
            'net_change': target_total - current_total,
            'target_total_vehicles': target_total,
            'total_vehicles': next_total,
        })

        current_counts = next_counts.copy()

    projected_market_share = projected_counts[vehicle_cols].div(projected_counts[vehicle_cols].sum(axis=1), axis=0).fillna(0)
    return projected_counts, projected_market_share, pd.DataFrame(projection_rows)

base_projected_counts, base_projected_market_share, base_projection_summary = rebuild_fleet(
    fleet_pivot,
    base_entry_share_future,
    base_exit_share_future,
    future_turnover_rate,
    target_total_fleet,
    vehicle_cols,
    FORECAST_END_YEAR,
)

calibrated_projected_counts, calibrated_projected_market_share, calibrated_projection_summary = rebuild_fleet(
    fleet_pivot,
    calibrated_entry_share_future,
    base_exit_share_future,
    future_turnover_rate,
    target_total_fleet,
    vehicle_cols,
    FORECAST_END_YEAR,
)

scenario_comparison = pd.DataFrame({
    'base_total_vehicles': base_projected_counts['total_vehicles'].loc[2021:2035],
    'calibrated_total_vehicles': calibrated_projected_counts['total_vehicles'].loc[2021:2035],
    'target_total_vehicles': target_total_fleet.loc[2021:2035],
    'base_electric_fleet_share': base_projected_market_share['electric'].loc[2021:2035],
    'calibrated_electric_fleet_share': calibrated_projected_market_share['electric'].loc[2021:2035],
    'base_hybrid_fleet_share': (base_projected_market_share['hev_sedan'] + base_projected_market_share['hev_suv']).loc[2021:2035],
    'calibrated_hybrid_fleet_share': (calibrated_projected_market_share['hev_sedan'] + calibrated_projected_market_share['hev_suv']).loc[2021:2035],
})
scenario_comparison.head()


## Step 11: Plot the Main Results

These plots help us answer three questions:

1. Does the calibrated model keep the total fleet on a smooth **population-linked** path?
2. Does it produce a smoother and more coherent sales mix?
3. Does the fleet market share evolve more gradually than the sales share, as expected under replacement dynamics?


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(base_projected_counts.index, base_projected_counts['total_vehicles'], label='Base total fleet')
ax.plot(calibrated_projected_counts.index, calibrated_projected_counts['total_vehicles'], label='Calibrated total fleet')
ax.axvline(last_observed_year, color='black', linestyle='--', alpha=0.7)
ax.set_title('Total Fleet Under Base and Full-Market Calibrated Scenarios')
ax.set_xlabel('Year')
ax.set_ylabel('Vehicles')
ax.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 12), sharex=True)
axes = axes.flatten()
for ax, col in zip(axes, vehicle_cols):
    ax.plot(base_projected_market_share.index, base_projected_market_share[col], label='Base fleet share')
    ax.plot(calibrated_projected_market_share.index, calibrated_projected_market_share[col], label='Calibrated fleet share')
    ax.axvline(last_observed_year, color='black', linestyle='--', alpha=0.7)
    ax.set_title(col)
    ax.set_ylabel('Fleet market share')
    ax.legend()
plt.suptitle('Fleet Market Share: Base vs Full-Market Calibrated Scenario', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(scenario_comparison.index, scenario_comparison['base_electric_fleet_share'], marker='o', label='Base EV fleet share')
ax.plot(scenario_comparison.index, scenario_comparison['calibrated_electric_fleet_share'], marker='s', label='Calibrated EV fleet share')
ax.plot(scenario_comparison.index, scenario_comparison['base_hybrid_fleet_share'], marker='o', linestyle='--', label='Base hybrid fleet share')
ax.plot(scenario_comparison.index, scenario_comparison['calibrated_hybrid_fleet_share'], marker='s', linestyle='--', label='Calibrated hybrid fleet share')
ax.set_title('Population-Linked Fleet Shares: Effect of Full-Market Calibration')
ax.set_xlabel('Year')
ax.set_ylabel('Fleet share')
ax.legend()
plt.show()

## Step 12: Final Summary Plots

These three plots summarize the calibrated scenario in the most direct way:

- calibrated sales share by type
- calibrated fleet market share by type
- calibrated total number of vehicles by type

They are useful for presentations and quick model checks because they separate the market input from the fleet-stock outcome.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.stackplot(
    calibrated_entry_share_future.index,
    [calibrated_entry_share_future[col] for col in vehicle_cols],
    labels=vehicle_cols,
    alpha=0.9,
)
ax.axvline(last_observed_year, color='black', linestyle='--', alpha=0.7)
ax.set_title('Population-Linked Calibrated Sales Share by Type')
ax.set_xlabel('Year')
ax.set_ylabel('Sales share')
ax.set_ylim(0, 1)
ax.legend(ncol=2, loc='upper right')
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.stackplot(
    calibrated_projected_counts.index,
    [calibrated_projected_counts[col] for col in vehicle_cols],
    labels=vehicle_cols,
    alpha=0.9,
)
ax.axvline(last_observed_year, color='black', linestyle='--', alpha=0.7)
ax.set_title('Population-Linked Calibrated Total Vehicle Counts by Type')
ax.set_xlabel('Year')
ax.set_ylabel('Number of vehicles')
ax.legend(ncol=2, loc='upper right')
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.stackplot(
    calibrated_projected_market_share.index,
    [calibrated_projected_market_share[col] for col in vehicle_cols],
    labels=vehicle_cols,
    alpha=0.9,
)
ax.axvline(last_observed_year, color='black', linestyle='--', alpha=0.7)
ax.set_title('Population-Linked Calibrated Fleet Market Share by Type')
ax.set_xlabel('Year')
ax.set_ylabel('Market share')
ax.set_ylim(0, 1)
ax.legend(ncol=2, loc='upper right')
plt.show()


In [ ]:
final_2035_summary = pd.DataFrame({
    'count_2035': calibrated_projected_counts.loc[2035, vehicle_cols],
    'market_share_2035': calibrated_projected_market_share.loc[2035, vehicle_cols],
}).sort_values('count_2035', ascending=False)
final_2035_summary


## Step 12: Key Tables to Inspect

These tables are useful when we want to discuss the calibration with others.

- `share_alignment`
  Historical alignment between the internal SAAQ entry shares and the external benchmark.

- `full_market_entry_share_comparison`
  Base vs calibrated future entry-share paths.

- `scenario_comparison`
  Main stock-level comparison through 2035.

In [ ]:
share_alignment.round(4).tail()

In [ ]:
full_market_entry_share_comparison.round(4).tail(10)

In [ ]:
scenario_comparison.round(4)

## Step 13: Save Outputs

If you want to reuse the calibrated results outside the notebook, run this cell.

The saved outputs are limited to the full-market calibration workflow, so the folder stays easier to navigate.

In [ ]:
output_dir = PROJECT_DIR / 'validation_outputs' / 'replacement_dynamics_ev_calibrated_model'
output_dir.mkdir(parents=True, exist_ok=True)

external_annual_benchmark.to_csv(output_dir / 'external_full_market_benchmark.csv')
share_alignment.to_csv(output_dir / 'saaq_vs_external_share_alignment.csv')
population_reference_summary.to_csv(output_dir / 'population_reference_summary.csv')
future_stock_controls.to_csv(output_dir / 'future_stock_controls.csv')
full_market_entry_share_comparison.to_csv(output_dir / 'full_market_entry_share_comparison.csv')
base_projected_counts.to_csv(output_dir / 'base_projected_counts.csv')
calibrated_projected_counts.to_csv(output_dir / 'calibrated_projected_counts.csv')
base_projected_market_share.to_csv(output_dir / 'base_projected_market_share.csv')
calibrated_projected_market_share.to_csv(output_dir / 'calibrated_projected_market_share.csv')
scenario_comparison.to_csv(output_dir / 'scenario_comparison.csv')

output_dir


## Summary of the Change

This notebook now calibrates the **whole future sales-mix curve**, and it no longer lets the total fleet collapse because of an unconstrained exit-rate fit.

### What stayed the same

- replacement-dynamics fleet accounting
- explicit entries and disposals
- external market calibration for all six vehicle categories
- gradual stock turnover

### What changed

- the external registrations file informs **all six vehicle categories**
- `2021` to `2025` use the observed mapped benchmark directly
- `2026` onward uses damped annual **share changes** for the full category vector
- net fleet growth is now linked to the Quebec population reference in [1710000901-eng.csv](/Users/natomanzolli/Downloads/1710000901-eng.csv)
- turnover volume is estimated from the SAAQ entry side instead of projecting exits independently

### Why this should behave better

- EV adoption is no longer calibrated in isolation
- hybrid and ICE categories move consistently with the same external market signal
- the new-sales mix is more realistic
- total vehicles evolve with a smooth population-linked path rather than an unrealistic structural decline
